 # Workflow for a transformation pathway of a single node energy system with perfect foresight

 In this application of the ETHOS.FINE framework, a transformation pathway of a energy system is modeled and optimized.

 All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

 The workflow is structures as follows:
 1. Required packages are imported and the input data path is set
 2. An energy system model instance is created
 3. Commodity sources are added to the energy system model
 4. Commodity conversion components are added to the energy system model
 5. Commodity storages are added to the energy system model
 6. Commodity sinks are added to the energy system model
 7. Material sinks are added to the energy system model
 8. Material sources are addeed to the energy system model
 9. Material conversions are added to the energy system model for recycling processes
 10. The energy system model is optimized
 11. Selected optimization results are presented


 # 1. Import required packages and set input data path

 The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd


cwd = Path.cwd()
data = getData()

 # 2. Create an energy system model instance

 The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

 The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {"GermanyRegion"}
commodityUnitsDict = {"electricity": r"GW$_{el}$"}
commodities = {"electricity"}
materials = {"lithium", "cobalt", "batteries_lithium_scrap", "batteries_cobalt_scrap"}
materialUnitsDict = {
    "lithium": "tons/h",
    "cobalt": "tons/h",
    "batteries_lithium_scrap": "tons/h",
    "batteries_cobalt_scrap": "tons/h",
}

numberOfTimeSteps = 4
hoursPerTimeStep = 8760 / 4

 # 2.1 define Transformation Pathway parameters

 Transformation Pathway Analyses can be run by setting a number of investment periods
 larger than 1, which is the default value and results in a single year optimization.

In [3]:
numberOfInvestmentPeriods = 3
startYear = 2020
interval = 5

In [4]:
pathwayBalanceLimit = pd.DataFrame(
    columns=["GermanyRegion", "Total", "lowerBound"],
    index=["Lithium_Resources", "Cobalt_Resources"],
)
pathwayBalanceLimit.loc["Lithium_Resources"] = [None, 54700, False]
pathwayBalanceLimit.loc["Cobalt_Resources"] = [None, 54700, False]

In [5]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    startYear=startYear,
    investmentPeriodInterval=interval,
    numberOfTimeSteps=4,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=8760 / 4,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    pathwayBalanceLimit=pathwayBalanceLimit,
)

 # 3. Add commodity sources to the energy system model

 ## 3.1. Electricity sources

 ### Wind onshore

 change weather conditions for the different investment periods

In [6]:
# operationRateMax = {}
# operationRateMax[2020] = 1.2 * data["Wind (onshore), operationRateMax"]
# operationRateMax[2025] = 0.7 * data["Wind (onshore), operationRateMax"]
# operationRateMax[2030] = 1 * data["Wind (onshore), operationRateMax"]

 define existing stock for wind onshore turbines

In [7]:
# stockWindonshoreCommissioning = {
#     2015: 10,
# }

# stockWindoffshoreCommissioning = {
#     2015: 15,
# }

 define invest and opex per capacity for wind onshore turbines

In [8]:
# investPerCapacityWind = {2015: 1.25, 2020: 1.1, 2025: 1, 2030: 0.95}

# opexPerCapacityWind = {
#     2015: 1.25 * 0.02,
#     2020: 1.1 * 0.02,
#     2025: 1 * 0.02,
#     2030: 0.95 * 0.02,
# }

 add wind onshore source to esM

In [9]:
esM.add(
    fn.Source(
        esM=esM,
        name="windonshore",
        commodity="electricity",
        hasCapacityVariable=True,
        # operationRateMax=data["Wind (onshore), operationRateMax"],
        operationRateMax=pd.Series([1, 0, 1, 0]),
        # capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=3.1,
        opexPerCapacity=3.1 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        # stockCommissioning=stockWindonshoreCommissioning,
    )
)

 Full load hours:

In [10]:
data["Wind (onshore), operationRateMax"].sum()

2300.4069071646272

 # 4. Add conversion components to the energy system model

 ### Electrolyzers

 add component with constant invest and opex per capacity

 # 5. Add commodity storages to the energy system model

 ## 5.1. Electricity storage

 ### Lithium ion batteries

 The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [11]:
esM.add(
    fn.Storage(
        esM=esM,
        name="batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=1,
        cyclicLifetime=10000,
        dischargeEfficiency=1,
        selfDischarge=0,
        chargeRate=1,
        dischargeRate=1,
        investPerCapacity={2020: 110, 2025: 100, 2030: 90},
        opexPerCapacity=0.002,
        interestRate=0.08,
        # chargeOpRateMax={2020:None, 2025:pd.DataFrame(data=[0]*4, columns=["GermanyRegion"]), 2030:pd.DataFrame(data=[0]*4, columns=["GermanyRegion"])},
        economicLifetime=5,
        materialIntensity={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 4.0}),
                "cobalt": pd.Series({"GermanyRegion": 4.0}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 3.9}),
                "cobalt": pd.Series({"GermanyRegion": 3.9}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 3.8}),
                "cobalt": pd.Series({"GermanyRegion": 3.8}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "lithium": pd.Series({"GermanyRegion": 0.7}),
                "cobalt": pd.Series({"GermanyRegion": 0.7}),
            },
            2025: {  # IP für 2025
                "lithium": pd.Series({"GermanyRegion": 0.8}),
                "cobalt": pd.Series({"GermanyRegion": 0.8}),
            },
            2030: {  # IP für 2030
                "lithium": pd.Series({"GermanyRegion": 0.9}),
                "cobalt": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

In [12]:
esM.add(
    fn.Storage(
        esM=esM,
        name="slbatteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=1,
        cyclicLifetime=10000,
        dischargeEfficiency=1,
        selfDischarge=0,
        chargeRate=1,
        dischargeRate=1,
        investPerCapacity={2020: 80, 2025: 70, 2030: 60},
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=5,
        # commissioningFix= {2020: 0, 2025: 10, 2030: 10},
        materialIntensity={
            2020: {  # IP für 2020
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 4.1}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 4.1}),
            },
            2025: {  # IP für 2025
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 4.0}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 4.0}),
            },
            2030: {  # IP für 2030
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 3.9}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 3.9}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 0.7}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.7}),
            },
            2025: {  # IP für 2025
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 0.8}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.8}),
            },
            2030: {  # IP für 2030
                "batteries_lithium_scrap": pd.Series({"GermanyRegion": 0.9}),
                "batteries_cobalt_scrap": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

 ## 5.2. Hydrogen storage

 ### Hydrogen filled salt caverns
 The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

 # 6. Add commodity sinks to the energy system model

 ## 6.1. Electricity sinks

 ### Electricity demand

 vary the demand with the years - increasing demand by 30% per year

In [13]:
electricityDemand = {}
electricityDemand[2020] = (1 + 0 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2025] = (1 + 1 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2030] = (1 + 2 * 0.3) * data["Electricity demand, operationRateFix"]

esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=8760 / 4,
    )
)

 ## 6.2. Hydrogen sinks

 ### Fuel cell electric vehicle (FCEV) demand

# 7. Add material sinks to the energy system model

In [14]:
esM.generationMaterialSinks()

Existing material sinks: set()
Missing materials sinks: {'cobalt', 'batteries_cobalt_scrap', 'batteries_lithium_scrap', 'lithium'}
New sink added: Cobalt demand
New sink added: Batteries_cobalt_scrap demand
New sink added: Batteries_lithium_scrap demand
New sink added: Lithium demand


# 8. Add material sources to the energy system model


## 8.1 Add primary material sources with pathway balance limit


In [15]:
esM.add(
    fn.Source(
        esM=esM,
        name="Lithium supply",
        hasCapacityVariable=True,
        commodity="lithium",
        pathwayBalanceLimitID="Lithium_Resources",
        investPerCapacity={2020: 100, 2025: 100, 2030: 100},
        opexPerCapacity=2,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Cobalt supply",
        hasCapacityVariable=True,
        commodity="cobalt",
        pathwayBalanceLimitID="Cobalt_Resources",
        investPerCapacity={2020: 100, 2025: 100, 2030: 100},
        opexPerCapacity=2,
    )
)

# esM.add(
#     fn.Storage(
#         esM=esM,
#         name="Slack supply",
#         hasCapacityVariable=False,
#         commodity="electricity",
#         opexPerChargeOperation=10000,
#     )
# )

In [16]:
# esM.generationSecondaryMaterialSources()

In [17]:
esM.add(
    fn.Source(
        esM=esM,
        name="slbatteries_batteries_cobalt_scrap_scrap",
        hasCapacityVariable=False,
        commodity="slbatteries_batteries_cobalt_scrap_scrap",
        material=True,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="slbatteries_batteries_lithium_scrap_scrap",
        hasCapacityVariable=False,
        commodity="slbatteries_batteries_lithium_scrap_scrap",
        material=True,
    )
)


esM.add(
    fn.Source(
        esM=esM,
        name="batteries_lithium_scrap",
        hasCapacityVariable=True,
        commodity="batteries_lithium_scrap",
        material=True,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="batteries_cobalt_scrap",
        hasCapacityVariable=True,
        commodity="batteries_cobalt_scrap",
        material=True,
    )
)

In [18]:
esM.add(
    fn.Sink(
        esM=esM,
        name="slbatteries_batteries_cobalt_scrap_scrap rec",
        hasCapacityVariable=False,
        commodity="slbatteries_batteries_cobalt_scrap_scrap",
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="slbatteries_batteries_lithium_scrap_scrap rec",
        hasCapacityVariable=False,
        commodity="slbatteries_batteries_lithium_scrap_scrap",
    )
)


esM.add(
    fn.Sink(
        esM=esM,
        name="batteries_lithium_scrap rec",
        hasCapacityVariable=False,
        commodity="batteries_lithium_scrap",
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="batteries_cobalt_scrap rec",
        hasCapacityVariable=False,
        commodity="batteries_cobalt_scrap",
    )
)

# 9. Add recycling plants

In [19]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler batteries",
        physicalUnit=r"tons/h",
        commodityConversionFactors={
            "batteries_lithium_scrap": -1,
            "lithium": 1,
            "batteries_cobalt_scrap": -1,
            "cobalt": 1,
        },
        hasCapacityVariable=True,
        economicLifetime=33,
    )
)


esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler batteries SL",
        physicalUnit=r"tons/h",
        commodityConversionFactors={
            "slbatteries_batteries_cobalt_scrap_scrap": -1,
            "lithium": 1,
            "slbatteries_batteries_lithium_scrap_scrap": -1,
            "cobalt": 1,
        },
        hasCapacityVariable=True,
        economicLifetime=33,
    )
)

 # 10. Optimize energy system model

 All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

In [20]:
# esM.aggregateTemporally(numberOfTypicalPeriods=30)

In [21]:
esM.optimize(timeSeriesAggregation=False, solver="gurobi")

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0064 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0071 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0040 sec)

Declaring shared potential constraint...
		(0.0004 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0019 sec)

Declaring material demand constraints...
LHS_demand op_srcSnk[GermanyRegion,Cobalt demand,0,0,0] + op_srcSnk[GermanyRegion,Cobalt demand,0,0,1] + op_srcSnk[GermanyRegion,Cobalt demand,0,0,2] + op_srcSnk[GermanyRegion,Cobalt demand,0,0,3]
RHS_demand 4.0*commis_stor[GermanyRegion,batteries,0]
LHS_demand op_srcSnk[GermanyRegion,Cobalt demand,1,0,0] + op_srcSnk[GermanyRegion,Cobalt d

 # 11. Selected results output

 ### Sources and Sink

 Show optimization summary

In [22]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of SourceSinkModel for year {year}")
    print(esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=year))


 Results of SourceSinkModel for year 2020
                                                   GermanyRegion
Component          Property        Unit                         
Cobalt demand      operation       [tons/h*h/a]           8760.0
                                   [tons/h*h]             8760.0
Cobalt supply      NPVcontribution [1e9 Euro]          72.887659
                   TAC             [1e9 Euro/a]        16.902949
                   capacity        [tons/h]                  1.0
                   capexCap        [1e9 Euro/a]        14.902949
                   commissioning   [tons/h]                  1.0
                   invest          [1e9 Euro]              100.0
                   operation       [tons/h*h/a]           8760.0
                                   [tons/h*h]             8760.0
                   opexCap         [1e9 Euro/a]              2.0
Electricity demand operation       [GW$_{el}$*h/a]        8760.0
                                   [GW$_{el}$*h

 ### Conversion

 Show optimization summary

In [23]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of ConversionMpdel for year {year}")
    print(esM.getOptimizationSummary("ConversionModel", outputLevel=2, ip=year))


 Results of ConversionMpdel for year 2020
Empty DataFrame
Columns: [GermanyRegion]
Index: []

 Results of ConversionMpdel for year 2025
                                              GermanyRegion
Component          Property      Unit                      
Recycler batteries capacity      [tons/h]          1.026484
                   commissioning [tons/h]          1.026484
                   operation     [tons/h*h/a]        2248.0
                                 [tons/h*h]          2248.0

 Results of ConversionMpdel for year 2030
                                                 GermanyRegion
Component             Property      Unit                      
Recycler batteries    capacity      [tons/h]           1.60274
                      commissioning [tons/h]          0.576256
                      operation     [tons/h*h/a]        3510.0
                                    [tons/h*h]          3510.0
Recycler batteries SL capacity      [tons/h]          1.956164
                   

 ### Storage

 Show optimization summary

In [24]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of StorageModel for year {year}")
    print(esM.getOptimizationSummary("StorageModel", outputLevel=2, ip=year))


 Results of StorageModel for year 2020
                                              GermanyRegion
Component Property           Unit                          
batteries NPVcontribution    [1e9 Euro]       260190.887116
          TAC                [1e9 Euro/a]      60339.339905
          capacity           [GW$_{el}$*h]           2190.0
          capexCap           [1e9 Euro/a]      60334.959905
          commissioning      [GW$_{el}$*h]           2190.0
          invest             [1e9 Euro]            240900.0
          operationCharge    [GW$_{el}$*h/a]         4380.0
                             [GW$_{el}$*h]           4380.0
          operationDischarge [GW$_{el}$*h/a]         4380.0
                             [GW$_{el}$*h]           4380.0
          opexCap            [1e9 Euro/a]              4.38

 Results of StorageModel for year 2025
                                               GermanyRegion
Component   Property           Unit                         
batteries   NPVcon

In [25]:
esM.pyM.timeSet.pprint()

timeSet : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     3 :    Any :   12 : {(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (1, 0, 0), (1, 0, 1), (1, 0, 2), (1, 0, 3), (2, 0, 0), (2, 0, 1), (2, 0, 2), (2, 0, 3)}


In [26]:
esM.getComponent("slbatteries").__dict__

{'name': 'slbatteries',
 'dimension': '1dim',
 'modelingClass': fine.storage.StorageModel,
 'hasCapacityVariable': True,
 'capacityVariableDomain': 'continuous',
 'capacityPerPlantUnit': 1,
 'processedCapacityPerPlantUnit': {0: 1, 1: 1, 2: 1},
 'hasIsBuiltBinaryVariable': False,
 'bigM': None,
 'partLoadMin': None,
 'economicLifetime': GermanyRegion    5.0
 dtype: float64,
 'technicalLifetime': GermanyRegion    5.0
 dtype: float64,
 'floorTechnicalLifetime': True,
 'ipTechnicalLifetime': GermanyRegion    1.0
 dtype: float64,
 'ipEconomicLifetime': GermanyRegion    1.0
 dtype: float64,
 'stockYears': [],
 'processedStockYears': [],
 'investPerCapacity': {2020: 80, 2025: 70, 2030: 60},
 'processedInvestPerCapacity': {0: GermanyRegion    80.0
  dtype: float64,
  1: GermanyRegion    70.0
  dtype: float64,
  2: GermanyRegion    60.0
  dtype: float64},
 'investIfBuilt': 0,
 'processedInvestIfBuilt': {0: GermanyRegion    0.0
  dtype: float64,
  1: GermanyRegion    0.0
  dtype: float64,
  2: G

In [27]:
esM.pyM.pprint()

67 Set Declarations
    DesignLocationComponentVarSet_conv : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    2 : {('GermanyRegion', 'Recycler batteries'), ('GermanyRegion', 'Recycler batteries SL')}
    DesignLocationComponentVarSet_srcSnk : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    5 : {('GermanyRegion', 'windonshore'), ('GermanyRegion', 'Lithium supply'), ('GermanyRegion', 'Cobalt supply'), ('GermanyRegion', 'batteries_lithium_scrap'), ('GermanyRegion', 'batteries_cobalt_scrap')}
    DesignLocationComponentVarSet_stor : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    2 : {('GermanyRegion', 'batteries'), ('GermanyRegion', 'slbatteries')}
    chargeOpConstrSet1_stor : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :  

In [28]:
esM.pyM.cap_stor.pprint()

cap_stor : Size=6, Index=designDimensionVarSet_stor
    Key                                 : Lower : Value             : Upper : Fixed : Stale : Domain
      ('GermanyRegion', 'batteries', 0) :     0 :            2190.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 1) :     0 : 999.9999999999629 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 2) :     0 :            2190.0 :  None : False : False : NonNegativeReals
    ('GermanyRegion', 'slbatteries', 0) :     0 :               0.0 :  None : False : False : NonNegativeReals
    ('GermanyRegion', 'slbatteries', 1) :     0 : 1190.000000000037 :  None : False : False : NonNegativeReals
    ('GermanyRegion', 'slbatteries', 2) :     0 :               0.0 :  None : False : False : NonNegativeReals


In [29]:
esM.pyM.chargeOp_stor.pprint()

chargeOp_stor : Size=24, Index=operationVarSet_stor*intraYearTimeSet
    Key                                       : Lower : Value             : Upper : Fixed : Stale : Domain
      ('GermanyRegion', 'batteries', 0, 0, 0) :     0 :            2190.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 0, 0, 1) :     0 :               0.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 0, 0, 2) :     0 :            2190.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 0, 0, 3) :     0 :               0.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 1, 0, 0) :     0 : 999.9999999999629 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 1, 0, 1) :     0 :               0.0 :  None : False : False : NonNegativeReals
      ('GermanyRegion', 'batteries', 1, 0, 2) :     0 : 999.9999999999629 :  None : False : False : NonNegativeReals
     